In [ ]:
import pandas as pd
import numpy as np
from math import sqrt
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


def load_and_clean(csv_path: str):
    print(f"[INFO] Loading data from {csv_path} ...")
    df = pd.read_csv(csv_path)
    df.columns = [col.strip() for col in df.columns]
    df.rename(columns={
        'Wind Speed (m/s)': 'WindSpeed',
        'Wind Direction (°)': 'WindDirection',
        'Theoretical_Power_Curve (KWh)': 'TheoreticalPower',
        'LV ActivePower (kW)': 'ActivePower'
    }, inplace=True)
    features = ['WindSpeed', 'WindDirection', 'TheoreticalPower']
    target = 'ActivePower'
    df.dropna(subset=features + [target], inplace=True)
    return df[features].values, df[target].values, features, target


def split_data(X, y, test_size=0.2, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


def evaluate_model(name, model, X_valid, y_valid):
    y_pred = model.predict(X_valid)
    rmse = sqrt(mean_squared_error(y_valid, y_pred))
    r2 = r2_score(y_valid, y_pred)
    print(f"[RESULT] {name} → RMSE: {rmse:.4f}, R²: {r2:.4f}")
    return {'Model': name, 'RMSE': rmse, 'R2': r2}


def train_rf(X_train, y_train):
    print("[INFO] Training Random Forest ...")
    rf = RandomForestRegressor(
        n_estimators=100, max_depth=20,
        min_samples_split=10, min_samples_leaf=2,
        n_jobs=-1, random_state=42
    )
    rf.fit(X_train, y_train)
    return rf


def train_lgbm(X_train, y_train, X_valid, y_valid):
    print("[INFO] Training LightGBM ...")
    dtrain = lgb.Dataset(X_train, label=y_train)
    dvalid = lgb.Dataset(X_valid, label=y_valid)
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'seed': 42,
        'n_jobs': -1
    }
    model = lgb.train(
        params, dtrain, num_boost_round=200,
        valid_sets=[dvalid],
        callbacks=[
            lgb.early_stopping(30),
            lgb.log_evaluation(50)
        ]
    )
    return model


def train_xgb(X_train, y_train, X_valid, y_valid):
    print("[INFO] Starting XGBoost training (no early stopping due to version compatibility) ...")
    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=150,
        learning_rate=0.05,
        eval_metric='rmse',
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    print("[INFO] XGBoost training complete.")
    return model



X, y, features, target = load_and_clean("Data/T1.csv")
X_train, X_valid, y_train, y_valid = split_data(X, y)

results = []

rf = train_rf(X_train, y_train)
results.append(evaluate_model("RandomForest", rf, X_valid, y_valid))

lgbm = train_lgbm(X_train, y_train, X_valid, y_valid)
results.append(evaluate_model("LightGBM", lgbm, X_valid, y_valid))

xgbm = train_xgb(X_train, y_train, X_valid, y_valid)
results.append(evaluate_model("XGBoost", xgbm, X_valid, y_valid))

print("\n=== SUMMARY ===")
summary = pd.DataFrame(results)
print(summary.to_string(index=False))
lgbm.save_model("wind_turbine_lgb.json")
xgbm.save_model('wind_turbine_xgb.json')



[INFO] Loading data from Data/T1.csv ...
[INFO] Training Random Forest ...
[RESULT] RandomForest → RMSE: 401.0151, R²: 0.9058
[INFO] Training LightGBM ...
Training until validation scores don't improve for 30 rounds
[50]	valid_0's rmse: 399.492
[100]	valid_0's rmse: 386.314
[150]	valid_0's rmse: 385.745
Early stopping, best iteration is:
[155]	valid_0's rmse: 385.535
[RESULT] LightGBM → RMSE: 385.5351, R²: 0.9129
[INFO] Starting XGBoost training (no early stopping due to version compatibility) ...
[INFO] XGBoost training complete.
[RESULT] XGBoost → RMSE: 386.7042, R²: 0.9124

=== SUMMARY ===
       Model       RMSE       R2
RandomForest 401.015058 0.905754
    LightGBM 385.535105 0.912890
     XGBoost 386.704204 0.912361


In [13]:
rf

RandomForestRegressor(max_depth=20, min_samples_leaf=2, min_samples_split=10,
                      n_jobs=-1, random_state=42)

In [ ]:
import importlib
import lgb2c
importlib.reload(lgb2c)
from lgb2c import LightGBMCExporter
import lightgbm as lgb
booster = lgb.Booster(model_file="wind_turbine_lgb.json")
exporter = LightGBMCExporter(booster)
result = exporter.export_header(r"2Board/LightGBM_model.h")
print(result)


In [40]:
import importlib
import xgb2c
importlib.reload(xgb2c)
from xgb2c import XGBoostCExporter
import xgboost as xgb

booster = xgb.Booster()
booster.load_model("wind_turbine_xgb.json")
exporter = XGBoostCExporter(booster)
result = exporter.export_header(r"2Board/XGBoost_model.h")
print(result)


Tree Stats:
  Total Trees        : 150
  Max Depth          : 7
  Min Depth          : 7
  Avg Depth          : 7.00
  Depth Std Dev      : 0.00
  Total Nodes        : 16448
  Max Nodes Per Tree : 127

Estimated Memory Usage: 385.50 KB



In [41]:
import numpy as np
inputs = 30
start = 2000
subset = X_valid[start:start + inputs]  # shape (30, num_features)
features = subset.shape[1]
print(f"#define INPUTS {inputs}")
print(f"#define FEATURES {features}")
print(f"float test_data[{inputs}][{features}] = {{")
for row in subset:
    row_str = ", ".join(f"{val:.8f}" for val in row)
    print(f"    {{{row_str}}},")
print("};")

#define INPUTS 30
#define FEATURES 3
float test_data[30][3] = {
    {8.11078262, 67.77146912, 1594.06118702},
    {11.17737007, 69.30249786, 3331.53595202},
    {3.26274800, 187.65559387, 29.93085714},
    {6.14069700, 329.94268799, 672.48912450},
    {11.35976982, 62.39863968, 3385.20832104},
    {3.04971910, 97.54427338, 17.78877012},
    {3.52549911, 204.25889587, 54.90791616},
    {10.26988029, 67.41493988, 2948.80702160},
    {6.74821520, 211.97039795, 906.56124796},
    {2.48409295, 89.32663727, 0.00000000},
    {7.13128901, 192.20480347, 1076.20203348},
    {2.90231299, 30.34938049, 0.00000000},
    {6.38450909, 76.27677155, 761.55077044},
    {3.89426589, 105.19550323, 108.59342706},
    {6.82627821, 254.69900513, 939.66544730},
    {7.30588102, 222.51739502, 1159.85398321},
    {4.38172817, 242.64390564, 197.37638004},
    {2.82118392, 66.26611328, 0.00000000},
    {7.39153194, 54.92308044, 1202.46386290},
    {1.36634302, 94.52765656, 0.00000000},
    {3.27262712, 90.09060669

In [42]:
xgbm.predict(X_valid[start:start+inputs])

array([1.38994678e+03, 2.77911353e+03, 1.50272923e+01, 5.71325745e+02,
       2.95114404e+03, 1.45380440e+01, 3.22970810e+01, 2.40854053e+03,
       7.38533813e+02, 9.22614861e+00, 9.71806885e+02, 8.94726086e+00,
       5.99543091e+02, 1.03840706e+02, 7.79285156e+02, 9.55053528e+02,
       1.40205582e+02, 1.71883309e+00, 1.05085046e+03, 9.22614861e+00,
       2.13526936e+01, 2.87965234e+03, 6.49585247e+00, 3.08518311e+03,
       2.72882861e+03, 6.02199768e+02, 1.35914612e+03, 1.40900464e+03,
       1.25496509e+03, 8.80675232e+02], dtype=float32)

In [43]:
lgbm.predict(X_valid[start:start+inputs])

array([1395.17328874, 2736.66358888,   11.18121746,  528.64974987,
       2972.52820398,    7.04974231,   25.35608425, 2427.39512295,
        752.80524862,    6.12139395,  951.10280235,    9.20494104,
        622.65097161,   88.23949014,  819.25222134, 1001.89130773,
        150.05046225,    5.86509467, 1026.89194668,    6.12139395,
         14.84085194, 2898.4132624 ,    5.19886472, 3095.53489272,
       2727.40709449,  574.67321822, 1337.94072327, 1394.81465596,
       1244.71509193,  875.95840986])